# Fingerprint: An End-to-End ML Framework for LLM Fingerprinting

---

## Notebook 10 — Transformer Baseline (DistilBERT)

---

### Purpose
Establish a deep learning baseline using a fine-tuned DistilBERT model.
This provides a principled comparison point between classical ML approaches
(Notebooks 03–09) and modern transformer-based sequence classification.

### Objectives
1. Load and tokenise the preprocessed text corpus
2. Fine-tune DistilBERT for multi-class LLM source classification
3. Evaluate with the same metrics used for classical models
4. Analyse per-class performance and confusion matrix
5. Compare against the best classical ML model from Notebook 09
6. Discuss the trade-off: accuracy vs resource cost

### Why DistilBERT?
| Property | Value |
|---|---|
| Architecture | Transformer encoder (6 layers, 66M params) |
| Size vs BERT | 40% smaller, 60% faster, 97% of BERT performance |
| Task | Sequence Classification (multi-class) |
| Input limit | 512 tokens (we use 256 for efficiency) |
| Why not BERT | DistilBERT is the right scale for a baseline |
| Why not RoBERTa | DistilBERT balances accuracy vs training time |

### Workflow
```
Raw Text (Fingerprint-Preserving pipeline)
        │
        ▼
  DistilBERT Tokenizer (max_length=256)
        │
        ▼
  HuggingFace Dataset
        │
        ▼
  Fine-tuning via HuggingFace Trainer API
        │
        ▼
  Evaluation → Classification Report, Confusion Matrix, ROC
        │
        ▼
  Comparison: DistilBERT vs Best Classical ML
```

### Notebook Outline
1. Imports
2. Configuration
3. Dataset Loading
4. Train/Val/Test Split
5. Tokenizer
6. Tokenise Dataset
7. Load Model
8. Build Trainer
9. Training Pipeline
10. Evaluation
11. Confusion Matrix
12. ROC Curves
13. Comparison with Classical ML
14. Model Saving
15. Notebook Summary

---

## 1. Imports

In [ ]:
import sys
import logging
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.transformer_trainer import TransformerTrainer
from src.evaluation.evaluator import ModelEvaluator
from src.visualization.plots import plot_confusion_matrix, plot_roc_curves
from src.utils.helpers import (
    set_global_seed, load_yaml, make_output_dirs,
    print_section_header, save_json,
)

print('✅ All libraries imported successfully.')

---

## 2. Configuration

In [ ]:
cfg = load_yaml(PROJECT_ROOT / 'configs' / 'training.yaml')

RANDOM_SEED  = cfg['random_seed']
TEST_SIZE    = cfg['evaluation']['test_size']
VAL_SIZE     = cfg['evaluation']['val_size']
TRANSFORMER_CFG = cfg['transformer']

set_global_seed(RANDOM_SEED)

DIR_MODELS  = PROJECT_ROOT / cfg['output']['models_dir']
DIR_FIGURES = PROJECT_ROOT / cfg['output']['figures_dir']
DIR_OUTPUTS = PROJECT_ROOT / cfg['output']['outputs_dir']
make_output_dirs(DIR_MODELS, DIR_FIGURES, DIR_OUTPUTS)

setup_logger = __import__('src.feature_engineering.utils', fromlist=['setup_logger']).setup_logger
setup_logger(str(PROJECT_ROOT / cfg['logging']['log_file']), cfg['logging']['level'])
logger = logging.getLogger(__name__)

print(f'Transformer model  : {TRANSFORMER_CFG["model_name"]}')
print(f'Max sequence length: {TRANSFORMER_CFG["max_length"]} tokens')
print(f'Batch size         : {TRANSFORMER_CFG["batch_size"]}')
print(f'Epochs             : {TRANSFORMER_CFG["num_epochs"]}')
print(f'Learning rate      : {TRANSFORMER_CFG["learning_rate"]}')
print(f'Device             : {TRANSFORMER_CFG["device"]}')

print()
print('⚠️  IMPORTANT: Fine-tuning on CPU is slow (hours for large datasets).')
print('   Set device: "cuda" in configs/training.yaml if a GPU is available.')

---

## 3. Dataset Loading

In [ ]:
# ── Load the Fingerprint-Preserving preprocessed dataset ──────────────────────
DATASET_PATH = PROJECT_ROOT / cfg['features']['tfidf']['fingerprint'].replace(
    'data/features/tfidf/tfidf_fingerprint.npz', 'data/processed/fingerprint/fingerprint_dataset.parquet'
)

# Fallback: try known path directly
PARQUET_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'processed' / 'fingerprint' / 'fingerprint_dataset.parquet',
    PROJECT_ROOT / 'data' / 'processed' / 'fingerprint_dataset.parquet',
]
dataset_path = next((p for p in PARQUET_CANDIDATES if p.exists()), None)

if dataset_path is None:
    raise FileNotFoundError(
        'Fingerprint-Preserving dataset not found. '
        'Run the Preprocessing pipeline (Notebook 00) first.'
    )

df = pd.read_parquet(dataset_path)
print(f'Dataset loaded: {len(df):,} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
df.head(3)

---

## 4. Train / Val / Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

TEXT_COL  = cfg.get('features', {}).get('text_col', 'generated_text')
LABEL_COL = cfg.get('features', {}).get('label_col', 'model_label')

# Auto-detect column names
TEXT_COL  = next((c for c in df.columns if 'text' in c.lower()), df.columns[0])
LABEL_COL = next((c for c in df.columns if 'label' in c.lower() or 'model' in c.lower()), df.columns[-1])

texts  = df[TEXT_COL].fillna('').tolist()
labels = df[LABEL_COL].tolist()

# Encode labels
le = LabelEncoder()
y  = le.fit_transform(labels)
CLASS_NAMES = le.classes_.tolist()

print(f'Text column  : {TEXT_COL}')
print(f'Label column : {LABEL_COL}')
print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')

# Train / Val / Test split
indices = list(range(len(texts)))
idx_tr_val, idx_te = train_test_split(
    indices, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_SEED
)
val_frac = VAL_SIZE / (1.0 - TEST_SIZE)
idx_tr, idx_val = train_test_split(
    idx_tr_val, test_size=val_frac,
    stratify=[y[i] for i in idx_tr_val],
    random_state=RANDOM_SEED,
)

train_texts  = [texts[i] for i in idx_tr]
train_labels = [y[i] for i in idx_tr]
val_texts    = [texts[i] for i in idx_val]
val_labels   = [y[i] for i in idx_val]
test_texts   = [texts[i] for i in idx_te]
test_labels  = [y[i] for i in idx_te]

print(f'\nTrain: {len(train_texts):,}  |  Val: {len(val_texts):,}  |  Test: {len(test_texts):,}')

---

## 5. Tokenizer

In [ ]:
# ── Initialise TransformerTrainer and load tokenizer ──────────────────────────
trainer_wrapper = TransformerTrainer(
    cfg=TRANSFORMER_CFG,
    num_labels=len(CLASS_NAMES),
    label_names=CLASS_NAMES,
)

tokenizer = trainer_wrapper.load_tokenizer()

print(f'Tokenizer vocabulary size : {tokenizer.vocab_size:,}')
print(f'Model max sequence length : {trainer_wrapper.max_seq_length} tokens')

# ── Inspect a sample tokenisation ─────────────────────────────────────────────
sample = train_texts[0]
tokens = tokenizer(sample, truncation=True, max_length=trainer_wrapper.max_seq_length)
print(f'\nSample text (first 100 chars): {sample[:100]}...')
print(f'Tokenised input IDs length   : {len(tokens["input_ids"])} tokens')

---

## 6. Tokenise Dataset

In [ ]:
# ── Tokenise train, val, and test sets ────────────────────────────────────────
print('Tokenising training set ...')
train_dataset = trainer_wrapper.tokenize(train_texts, train_labels)

print('Tokenising validation set ...')
val_dataset = trainer_wrapper.tokenize(val_texts, val_labels)

print('Tokenising test set ...')
test_dataset = trainer_wrapper.tokenize(test_texts, test_labels)

print(f'\nTrain dataset : {train_dataset}')
print(f'Val dataset   : {val_dataset}')
print(f'Test dataset  : {test_dataset}')

---

## 7. Load Model

In [ ]:
# ── Load DistilBERT with classification head ───────────────────────────────────
model = trainer_wrapper.load_model()

# Model parameter count
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Model           : {trainer_wrapper.model_name}')
print(f'Total parameters: {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Classification head classes: {trainer_wrapper.num_labels}')

---

## 8. Build Trainer

In [ ]:
# ── Build HuggingFace Trainer with config-driven TrainingArguments ─────────────
hf_trainer = trainer_wrapper.build_trainer(
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print('HuggingFace Trainer built successfully.')
print(f'Output directory  : {TRANSFORMER_CFG["output_dir"]}')
print(f'Training epochs   : {TRANSFORMER_CFG["num_epochs"]}')
print(f'Batch size        : {TRANSFORMER_CFG["batch_size"]}')
print(f'Learning rate     : {TRANSFORMER_CFG["learning_rate"]}')
print(f'Weight decay      : {TRANSFORMER_CFG["weight_decay"]}')
print(f'Warmup ratio      : {TRANSFORMER_CFG["warmup_ratio"]}')

---

## 9. Training Pipeline

In [ ]:
# ── Fine-tune DistilBERT ───────────────────────────────────────────────────────
#
# ⚠️  This cell will take significant time on CPU:
#     - CPU  : ~30–90 min depending on dataset size and epoch count
#     - GPU  : ~5–15 min
#
# To speed up training:
#   1. Reduce num_epochs in configs/training.yaml
#   2. Reduce the dataset size (stratified subsample)
#   3. Use a GPU (set device: cuda in configs/training.yaml)

print_section_header('Fine-tuning DistilBERT')
print(f'  Epochs : {trainer_wrapper.num_epochs}')
print(f'  Device : {trainer_wrapper.device}')
print()

train_result = trainer_wrapper.train()

print(f'\n✅ Training complete.')
print(f'   Training time   : {trainer_wrapper.train_time_:.1f}s')
print(f'   Training samples: {train_result.training_loss:.4f} (final loss)')

In [ ]:
# ── Plot training loss curve (if trainer logs are available) ───────────────────
try:
    log_history = hf_trainer.state.log_history
    train_logs  = [x for x in log_history if 'loss' in x and 'eval_loss' not in x]
    eval_logs   = [x for x in log_history if 'eval_loss' in x]

    if train_logs:
        steps  = [x['step'] for x in train_logs]
        losses = [x['loss'] for x in train_logs]

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=steps, y=losses, mode='lines', name='Train Loss'))

        if eval_logs:
            eval_steps  = [x['step']      for x in eval_logs]
            eval_losses = [x['eval_loss'] for x in eval_logs]
            fig.add_trace(go.Scatter(x=eval_steps, y=eval_losses,
                                     mode='lines+markers', name='Val Loss'))

        fig.update_layout(
            title='DistilBERT Fine-tuning — Training & Validation Loss',
            xaxis_title='Step',
            yaxis_title='Loss',
            template='plotly_dark',
        )
        fig.show()
except Exception as e:
    print(f'Could not plot loss curve: {e}')

---

## 10. Evaluation

In [ ]:
# ── Generate predictions on the held-out test set ─────────────────────────────
import time as _time

print('Generating predictions on test set ...')
t0 = _time.perf_counter()
y_pred, y_true = trainer_wrapper.predict(test_dataset)
pred_time = _time.perf_counter() - t0

print(f'Prediction time: {pred_time:.2f}s  ({len(y_pred)} samples)')

# ── Compute metrics ────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)

acc           = accuracy_score(y_true, y_pred)
prec_macro    = precision_score(y_true, y_pred, average='macro', zero_division=0)
rec_macro     = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_macro      = f1_score(y_true, y_pred, average='macro', zero_division=0)
f1_weighted   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

transformer_metrics = {
    'model_name':        'distilbert',
    'feature_set':       'raw_text',
    'accuracy':          round(acc, 6),
    'precision_macro':   round(prec_macro, 6),
    'recall_macro':      round(rec_macro, 6),
    'f1_macro':          round(f1_macro, 6),
    'f1_weighted':       round(f1_weighted, 6),
    'train_time_s':      round(trainer_wrapper.train_time_, 4),
    'pred_time_s':       round(pred_time, 4),
}

print('\nDistilBERT Evaluation Results:')
for k, v in transformer_metrics.items():
    if isinstance(v, float):
        print(f'  {k:22s}: {v:.4f}')

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

---

## 11. Confusion Matrix

In [ ]:
cm_transformer = confusion_matrix(y_true, y_pred)

cm_path = DIR_FIGURES / 'distilbert_confusion_matrix.png'
plot_confusion_matrix(
    cm=cm_transformer,
    class_names=CLASS_NAMES,
    title='DistilBERT — Confusion Matrix (Test Set)',
    out_path=cm_path,
    normalize=True,
)
print(f'✅ Confusion matrix saved → {cm_path.name}')

---

## 12. ROC Curves

In [ ]:
# ── Extract softmax probabilities for ROC computation ─────────────────────────
import torch
from scipy.special import softmax as scipy_softmax

# Re-run inference to get logits (already have y_pred from trainer.predict)
raw_preds = hf_trainer.predict(test_dataset)
logits    = raw_preds.predictions
y_proba   = scipy_softmax(logits, axis=1)

roc_path = DIR_FIGURES / 'distilbert_roc_curves.png'
plot_roc_curves(
    y_test=y_true,
    y_proba=y_proba,
    class_names=CLASS_NAMES,
    title='DistilBERT — One-vs-Rest ROC Curves',
    out_path=roc_path,
)
print(f'✅ ROC curves saved → {roc_path.name}')

# Compute and display ROC-AUC
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

y_bin    = label_binarize(y_true, classes=list(range(len(CLASS_NAMES))))
roc_auc  = roc_auc_score(y_bin, y_proba, multi_class='ovr', average='macro')
transformer_metrics['roc_auc_macro'] = round(roc_auc, 6)
print(f'ROC-AUC (macro): {roc_auc:.4f}')

---

## 13. Comparison with Classical ML

In [ ]:
# ── Load best classical ML metrics from Notebook 09 ───────────────────────────
import json

best_selection_path = DIR_OUTPUTS / 'best_model_selection.json'

if best_selection_path.exists():
    with open(best_selection_path) as f:
        best_classical = json.load(f)
else:
    # Placeholder if NB09 has not been run
    best_classical = {
        'selected_model': 'Best Classical ML (placeholder)',
        'f1_macro': 0.0,
        'f1_weighted': 0.0,
        'roc_auc_macro': 0.0,
        'pred_time_s': 0.0,
    }
    print('⚠️  best_model_selection.json not found. Run Notebook 09 first.')

# ── Build comparison DataFrame ────────────────────────────────────────────────
comparison_rows = [
    {
        'Model':       best_classical.get('selected_model', 'Best Classical ML'),
        'Type':        'Classical ML',
        'Feature Set': best_classical.get('feature_set', 'N/A'),
        'Macro F1':    best_classical.get('f1_macro', 0),
        'Weighted F1': best_classical.get('f1_weighted', 0),
        'ROC-AUC':     best_classical.get('roc_auc_macro', 0),
        'Train Time (s)': None,
        'Pred Time (s)':  best_classical.get('pred_time_s', 0),
    },
    {
        'Model':       'DistilBERT (fine-tuned)',
        'Type':        'Transformer',
        'Feature Set': 'raw_text',
        'Macro F1':    transformer_metrics['f1_macro'],
        'Weighted F1': transformer_metrics['f1_weighted'],
        'ROC-AUC':     transformer_metrics.get('roc_auc_macro', None),
        'Train Time (s)': transformer_metrics['train_time_s'],
        'Pred Time (s)':  transformer_metrics['pred_time_s'],
    },
]
comp_df = pd.DataFrame(comparison_rows)
comp_df

In [ ]:
# ── Visual comparison ──────────────────────────────────────────────────────────
fig = px.bar(
    comp_df.melt(
        id_vars='Model',
        value_vars=['Macro F1', 'Weighted F1', 'ROC-AUC'],
    ),
    x='variable',
    y='value',
    color='Model',
    barmode='group',
    title='Classical ML vs DistilBERT — Key Metrics Comparison',
    template='plotly_dark',
    labels={'variable': 'Metric', 'value': 'Score'},
)
fig.update_yaxes(range=[0, 1.05])
fig.show()

# ── Trade-off: Accuracy vs Training Time ──────────────────────────────────────
fig2 = px.scatter(
    comp_df,
    x='Train Time (s)',
    y='Macro F1',
    color='Type',
    text='Model',
    size_max=20,
    title='Accuracy vs Training Cost — Classical ML vs Transformer',
    template='plotly_dark',
)
fig2.update_traces(textposition='top center')
fig2.show()

In [ ]:
# ── Analysis: Which model wins? ────────────────────────────────────────────────
classical_f1   = best_classical.get('f1_macro', 0)
transformer_f1 = transformer_metrics['f1_macro']
delta          = transformer_f1 - classical_f1

print('=' * 60)
print('  Classical ML vs DistilBERT — Summary')
print('=' * 60)
print(f'  Classical ML Macro F1  : {classical_f1:.4f}')
print(f'  DistilBERT Macro F1    : {transformer_f1:.4f}')
print(f'  Δ Macro F1             : {delta:+.4f}')
print()
if delta > 0.01:
    print('  → DistilBERT outperforms Classical ML by a meaningful margin.')
    print('    The transformer captures complex sequential patterns that bag-of-words models miss.')
elif delta > 0:
    print('  → DistilBERT marginally outperforms Classical ML.')
    print('    The benefit may not justify the substantially higher training cost.')
else:
    print('  → Classical ML is competitive with or outperforms DistilBERT.')
    print('    For this dataset, feature engineering captures LLM fingerprints effectively.')
    print('    The simpler model is preferred given significantly lower training cost.')
print('=' * 60)

---

## 14. Model Saving

In [ ]:
# ── Save fine-tuned DistilBERT ─────────────────────────────────────────────────
trainer_wrapper.save(out_dir=str(DIR_MODELS / 'distilbert_fingerprint'))
print(f'✅ DistilBERT model saved → {DIR_MODELS / "distilbert_fingerprint"}')

# Save transformer metrics
save_json(transformer_metrics, DIR_OUTPUTS / 'distilbert_metrics.json')
print('✅ DistilBERT metrics saved → outputs/distilbert_metrics.json')

---

## 15. Notebook Summary

### DistilBERT Baseline Results

| Metric | DistilBERT | Best Classical ML | Winner |
|---|---|---|---|
| Accuracy | *(run to populate)* | *(run to populate)* | *(TBD)* |
| Macro F1 | *(run to populate)* | *(run to populate)* | *(TBD)* |
| Weighted F1 | *(run to populate)* | *(run to populate)* | *(TBD)* |
| ROC-AUC | *(run to populate)* | *(run to populate)* | *(TBD)* |
| Training Time | *(run to populate)* | *(run to populate)* | *(TBD)* |
| Pred Time | *(run to populate)* | *(run to populate)* | *(TBD)* |

### Key Takeaways
- DistilBERT provides a strong semantic understanding baseline
- Training cost is significantly higher than classical ML
- Whether the accuracy gain (if any) justifies the cost depends on deployment constraints
- Classical ML + TF-IDF remains competitive for LLM fingerprinting

→ **Notebook 11**: Final Evaluation — integrate all results for overall conclusions

---
*Fingerprint Project — Transformer Baseline (DistilBERT) — Complete*